# 🌐 Autonomous Agentic Bioinformatics: The Universal `embpy` MCP Pipeline

In modern bioinformatics, foundation models (e.g., scGPT, Geneformer, ESM-2, Borzoi, scVI) transform high-dimensional biological measurements into actionable vector embeddings. However, connecting these powerful pipelines directly to Large Language Models (LLMs) like Claude, GPT-4, or Cursor presents a major bottleneck: **Context Window Exhaustion**.

A typical single-cell expression matrix with 10,000 cells and 2,000 genes exceeds **20,000,000 tokens**—far beyond any model's context capacity.

### The Solution: `embpy`'s Universal Model Context Protocol (MCP) Server
`embpy` implements a **native MCP server** operating on strict **Token-Saving Guardrails**:
1. **Never Return Raw Data:** High-dimensional arrays stay in `.h5ad` files or memory caches on disk.
2. **Compact Metadata Responses:** Tools return concise JSON envelopes detailing execution status, file paths, array shapes (e.g., `(400, 512)`), feature mapping rates, and Markdown summary tables.
3. **Graceful Error Trapping:** Catch library warnings and shape mismatches and return actionable error strings so the agent can self-correct without crashing.

---

### The 4-Pillar Universal State Machine
The server organizes the entire universal pipeline across 9 autonomous tools:
* **Pillar A: Inspection & Preprocessing:** `inspect_data`, `resolve_identifiers`, `preprocess_counts`
* **Pillar B: Universal BioEmbedder:** `list_model_registry`, `compute_embeddings`
* **Pillar C: Metadata Annotation:** `annotate_metadata`
* **Pillar D: Analysis & Benchmarking:** `compute_umap`, `benchmark_scib`, `benchmark_perturbations`


In [ ]:
import os
import sys
import tempfile
import numpy as np
import pandas as pd
from anndata import AnnData
import scipy.sparse as sp

# Ensure local embpy source tree is loaded
for candidate in ["src", "../src", os.path.join(os.getcwd(), "src"), os.path.join(os.path.dirname(os.getcwd()), "src")]:
    if os.path.isdir(candidate):
        full_path = os.path.abspath(candidate)
        if full_path not in sys.path:
            sys.path.insert(0, full_path)
        break

import embpy
from embpy.mcp import (
    inspect_data,
    resolve_identifiers,
    preprocess_counts,
    list_model_registry,
    compute_embeddings,
    annotate_metadata,
    compute_umap,
    benchmark_scib,
    benchmark_perturbations,
    create_server,
)

print(f"Loaded embpy version: {embpy.__version__}")


## 1. Creating a Sample Single-Cell Dataset

To simulate a real-world workflow, we generate a synthetic single-cell dataset with:
- Raw integer count distributions
- Batch confounding (`batch_1`, `batch_2`)
- Multiple cell types (`T_cell`, `B_cell`, `Monocyte`)
- Perturbation conditions (`ctrl`, `drug_A`)
- Real HGNC gene symbols (`TP53`, `GAPDH`, `EGFR`, etc.)


In [ ]:
# Set up a temporary working directory
work_dir = tempfile.mkdtemp(prefix="embpy_universal_agent_")
raw_data_path = os.path.join(work_dir, "raw_dataset.h5ad")

np.random.seed(42)
n_cells = 60
n_genes = 20

# Gene symbols
gene_symbols = [
    "TP53", "GAPDH", "ACTB", "EGFR", "MYC",
    "CD4", "CD8A", "IL2", "TNF", "VEGFA",
    "INS", "ALB", "BRCA1", "PTEN", "APOE",
    "MTOR", "STAT3", "MAPK1", "KRAS", "ESR1"
]

# Simulate raw UMI count matrix
X = np.random.negative_binomial(5, 0.3, size=(n_cells, n_genes)).astype(np.float32)

# Cell annotations
cell_types = np.random.choice(["T_cell", "B_cell", "Monocyte"], size=n_cells)
batches = np.random.choice(["batch_1", "batch_2"], size=n_cells)
perturbations = np.random.choice(["ctrl", "drug_A"], size=n_cells)
splits = np.random.choice(["real", "pred"], size=n_cells)

obs = pd.DataFrame({
    "cell_type": cell_types,
    "batch": batches,
    "perturbation": perturbations,
    "split": splits,
}, index=[f"cell_{i:03d}" for i in range(n_cells)])

var = pd.DataFrame(index=gene_symbols)

adata_raw = AnnData(X=X, obs=obs, var=var)
adata_raw.write_h5ad(raw_data_path)

print(f"Created sample raw AnnData dataset at: {raw_data_path}")
print(f"Dimensions: {n_cells} cells x {n_genes} genes")


## 2. Pillar A: Inspection & Preprocessing Skills

An autonomous agent starts by profiling the unknown dataset using `inspect_data`.
Notice that instead of dumping the matrix into the context window, `inspect_data` returns:
- Matrix shape `(n_cells, n_genes)`
- Layers and observation columns
- Log-normalization status detection
- Automated candidate recommendations for `label_key` and `batch_key`


In [ ]:
# Step 1: Autonomous Data Inspection
inspect_res = inspect_data(raw_data_path)
print("=== inspect_data Output ===")
for k, v in inspect_res.items():
    print(f"  {k}: {v}")


### Resolving Identifiers (`resolve_identifiers`)
Foundation models require specific vocabularies (e.g. Ensembl IDs for scGPT/Geneformer, or SMILES for molecular models).
`resolve_identifiers` automatically validates and maps identifiers, returning success rates and sample pairs without filling the prompt with thousands of IDs.


In [ ]:
# Step 2: Harmonize Gene Symbols to Ensembl IDs
resolved_h5ad_path = os.path.join(work_dir, "resolved_dataset.h5ad")

resolve_res = resolve_identifiers(
    path=raw_data_path,
    entity_type="gene",
    target_id_type="ensembl",
    organism="human",
    update_h5ad_path=resolved_h5ad_path,
)

print("=== resolve_identifiers Output ===")
print("Status:", resolve_res["status"])
print("Total genes submitted:", resolve_res["total_submitted"])
print("Mapped count:", resolve_res["mapped_count"])
print("Mapping success rate:", f"{resolve_res['mapping_success_rate']}%")
print("Sample mapped pairs:", resolve_res["sample_mapping"])


### Preprocessing Counts (`preprocess_counts`)
Single-cell models require normalized expression matrices.
`preprocess_counts` performs:
1. Cell & gene quality filtering
2. Library size normalization (`sc.pp.normalize_total` to `1e4`)
3. Log1p transformation (`sc.pp.log1p`)
4. Highly variable gene (HVG) selection

Crucially, **the expression matrix is preserved on disk** and stored under `.layers['log_normalized']` and `.layers['counts']`. The tool returns only the preprocessed file path and compact summary metrics.


In [ ]:
# Step 3: Run Automated Preprocessing
preprocessed_path = os.path.join(work_dir, "preprocessed.h5ad")

prep_res = preprocess_counts(
    path=resolved_h5ad_path,
    output_path=preprocessed_path,
    min_genes=1,
    min_cells=1,
    target_sum=1e4,
    n_top_genes=15,
    select_hvg=True,
    log_transform=True,
)

print("=== preprocess_counts Output ===")
print("Status:", prep_res["status"])
print("Output Path:", prep_res["output_path"])
print("Final Cell Count:", prep_res["n_cells"])
print("Final Gene Count:", prep_res["n_genes"])
print("HVG Count:", prep_res["hvg_count"])
print("Layers saved:", prep_res["layers"])


## 3. Pillar B: Universal BioEmbedder Skills

Next, the agent discovers available models and executes embedding generation.
- `list_model_registry`: Reports ~150 biological foundation models across modalities (single cell, gene, protein, small molecule).
- `compute_embeddings`: Wraps `BioEmbedder.embed()` to extract multi-model latent representations and stores them directly in `.obsm` on disk.


In [ ]:
# Step 4: Query Model Registry
registry_info = list_model_registry()
print("=== list_model_registry Output ===")
print("Hardware Device:", registry_info["device"])
print("Total Models in Registry:", registry_info["total_models"])
print("Supported Modalities:", registry_info["supported_entities"])
print("Single-cell models sample:", registry_info["all_singlecell_models"][:6])


In [ ]:
# Step 5: Compute Universal Embeddings (PCA baseline)
embedded_path = os.path.join(work_dir, "embedded.h5ad")

embed_res = compute_embeddings(
    path=preprocessed_path,
    models=["pca"],
    output_path=embedded_path,
    entity_type="cell",
    device="cpu",
    batch_size=32,
    preprocessing="none",  # already preprocessed in Step 3
)

print("=== compute_embeddings Output ===")
print("Status:", embed_res["status"])
print("Output Path:", embed_res["output_path"])
print("Models Computed:", embed_res["models_computed"])
print("Keys added to .obsm:", embed_res["embedding_keys"])
print("Array Shapes (compact metadata):", embed_res["embedding_shapes"])


## 4. Pillar C: Annotation Skills

The `annotate_metadata` tool enriches AnnData objects with external biological knowledge:
- Gene Ontology (GO) biological processes
- Functional pathways (KEGG, Reactome)
- Disease and pharmacology associations

It updates `.obs` or `.uns` on disk and returns only the newly added annotation column names.


In [ ]:
# Step 6: Enrich Dataset with External Biological Knowledge
annotated_path = os.path.join(work_dir, "annotated.h5ad")

annot_res = annotate_metadata(
    path=embedded_path,
    annotation_type="pathway",
    entity_type="gene",
    output_path=annotated_path,
)

print("=== annotate_metadata Output ===")
print("Status:", annot_res["status"])
print("Annotation Category:", annot_res["annotation_type"])
print("Entities Annotated:", annot_res["entities_annotated"])
print("Added Annotation Columns:", annot_res["added_columns"])


## 5. Pillar D: Analysis & Benchmarking Skills

Once embeddings are computed and metadata enriched, the autonomous agent analyzes and benchmarks representation quality:
1. `compute_umap`: Generates a kNN graph and calculates 2D UMAP projection coordinates.
2. `benchmark_scib`: Evaluates biological conservation (NMI, ARI, ASW label) and batch correction (ASW batch, iLISI) using scIB metrics.
3. `benchmark_perturbations`: Evaluates perturbation shift predictions using ArcInstitute `cell-eval`.


In [ ]:
# Step 7: Compute 2D Latent UMAP
umap_path = os.path.join(work_dir, "umap_projected.h5ad")

umap_res = compute_umap(
    path=annotated_path,
    embedding_key="X_pca",
    output_path=umap_path,
    n_neighbors=5,
)

print("=== compute_umap Output ===")
print("Status:", umap_res["status"])
print("UMAP Key in .obsm:", umap_res["umap_key"])
print("UMAP Coordinate Shape:", umap_res["umap_shape"])
print("Cell Count:", umap_res["n_cells"])


In [ ]:
# Step 8: Benchmark scIB Bio-Conservation & Batch Correction
scib_csv = os.path.join(work_dir, "scib_scores.csv")

scib_res = benchmark_scib(
    path=umap_path,
    embedding_keys=["X_pca"],
    label_key="cell_type",
    batch_key="batch",
    output_csv=scib_csv,
)

print("=== benchmark_scib Output ===")
print("Status:", scib_res["status"])
print("Rankings:", scib_res["rankings"])
print("\nMarkdown Summary Table for Agent Context:\n")
print(scib_res["markdown_table"])


In [ ]:
# Step 9: Benchmark Perturbation Effects with cell-eval
pert_csv = os.path.join(work_dir, "pert_scores.csv")

pert_res = benchmark_perturbations(
    path=umap_path,
    control_pert="ctrl",
    pert_col="perturbation",
    split_col="split",
    embedding_keys=["X_pca"],
    profile="minimal",
    output_csv=pert_csv,
)

print("=== benchmark_perturbations Output ===")
print("Status:", pert_res["status"])
print("Control Perturbation:", pert_res["control_pert"])
print("Perturbation Count:", pert_res["n_perturbations"])
print("\nMarkdown Differential Metrics Table:\n")
print(pert_res["markdown_table"])


## 6. Token-Saving Guardrail Analysis

Why is this architecture essential for autonomous AI coding agents?
Below is a comparison of token consumption between returning raw arrays vs. returning `embpy` MCP metadata:


In [ ]:
import json

# Scenario A: Naive approach returning raw expression & embedding matrices
naive_payload = {
    "expression_matrix": adata_raw.X.tolist(),
    "pca_coordinates": np.random.randn(n_cells, 50).tolist(),
    "umap_coordinates": np.random.randn(n_cells, 2).tolist(),
}
naive_json_str = json.dumps(naive_payload)
naive_estimated_tokens = len(naive_json_str) // 4

# Scenario B: embpy Token-Saving MCP Envelope
mcp_payload = {
    "status": "success",
    "preprocessed_path": preprocessed_path,
    "embedding_keys": ["X_pca"],
    "shapes": {"X_pca": [n_cells, 50]},
    "rankings": ["X_pca"],
    "bio_conservation": 0.82,
}
mcp_json_str = json.dumps(mcp_payload)
mcp_estimated_tokens = len(mcp_json_str) // 4

print("📊 Context Window Token Consumption:")
print(f"  • Naive Raw Matrix Return: ~{naive_estimated_tokens:,} tokens (Scales exponentially with N cells/genes)")
print(f"  • embpy MCP Token-Saving Return: ~{mcp_estimated_tokens:,} tokens (Constant O(1) complexity)")
print(f"  • Token Reduction: {((naive_estimated_tokens - mcp_estimated_tokens) / naive_estimated_tokens) * 100:.1f}% savings!")


## 7. Conclusion & Next Steps

You have executed the full 4-pillar universal embedding pipeline:
1. **Inspected** raw data without context overflow.
2. **Resolved** biological entity identifiers to official Ensembl standards.
3. **Preprocessed** counts (`sc.pp.normalize_total`, `sc.pp.log1p`, HVGs) writing `.h5ad` to disk.
4. **Discovered** foundation models and extracted representations.
5. **Annotated** biological metadata.
6. **Analyzed** latent space via UMAP and **benchmarked** bio-conservation and perturbations with scIB and cell-eval.

To integrate this directly into your IDE (Cursor, Claude Desktop, Windsurf), see the client configuration guide in `docs/mcp/client-setup.md`!
